
> ## 📊 결론 — Y타겟 비교: **whiff%만 예측 가능** (주력 타겟 확정)
>
> 경기단위 타겟 4종에 정형 baseline + 영상 융합을 paired t-test로 비교.
>
> | Y | 정형 R² | 분류 AUC | 영상 융합 |
> |---|---|---|---|
> | **y_whiff** ⭐ | 0.036 | **0.637** | 무의 |
> | y_xwoba | 0.002 | 0.553 | 무의 |
> | y_fip | -0.004 | 0.526 | 무의 |
> | y_era | 0.009 | 0.569 | 유의하게 나빠짐 |
>
> - **whiff%가 유일하게 예측 가능** → 주력 타겟 확정. (수비·운이 섞인 xwoba·fip·era는 사실상 예측 불가)
> - **"다른 Y면 영상이 살까?" 가설 기각** — 4종 모두 무의/음수. 이후 제구 타겟(zone%·ball%)까지 확장했으나 역시 무의(→ 12번 E8).
> - R²가 낮은 건 한계가 아니라 **문제의 천장** (학계도 구속→ERA R²≈0.05~0.08).
> - → 값 예측 대신 **분류로 전환** (→ 19번, AUC 0.65).

# 18. 타겟(Y)별 정형 baseline + 영상 융합 비교

**목적**: "영상 생체역학이 *어떤 퍼포먼스 지표*와 연결되는가"를 4개 Y로 엄밀 비교.
whiff%로만 봤을 땐 무의했으나, 타구질(xwoba)·자력(fip)·실점(era) 등 다른 면에서 영상이 살아나는지 검증.

| Y | 방향 | 표본 |
|---|---|---|
| y_whiff | ↑ 높을수록 좋음 | 스윙 20+ |
| y_xwoba | ↓ 낮을수록 좋음 | 인플레이 타구 있는 경기 |
| y_fip | ↓ 낮을수록 좋음 | (극단값 → 클리핑) |
| y_era | ↓ 낮을수록 좋음 | (극단값 → 클리핑) |

**엄밀성 장치**
1. 정형 vs 융합을 **동일 경기·동일 seed·동일 파라미터**로 비교 (paired)
2. 타겟별 **결측 제거 후** 공통 행으로만 평가 (Y마다 유효 경기 다름)
3. FIP/ERA **이상치 클리핑**(1~99%) — 경기단위 극단값 방어
4. **다중검정 보정**(Bonferroni, 4개 타겟) — p값 부풀림 방지
5. R²(↑)·RMSE(↓) 동시 기록, 방향 혼동 방지
6. seed 고정, 결과를 메타와 함께 `4_output/y_target_comparison.csv`

## 1. 설정 + 경로

In [ ]:
import os
import numpy as np
import pandas as pd
from itertools import product

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/MLB_pitcher'
    DATA = f'{BASE}/data'
    # 정형 피처 위치 자동탐색 (features / 4_features)
    STAT_DIR = next((d for d in [f'{DATA}/features', f'{DATA}/4_features']
                     if os.path.exists(f'{d}/features_pitch15.parquet')), f'{DATA}/features')
    BIO_DIR  = f'{DATA}/4_features'
    TARGETS_PATH = f'{DATA}/game_targets.parquet'
    OUTPUT_DIR = f'{BASE}/4_output'
else:
    BASE = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'
    STAT_DIR = os.path.join(BASE, '0_data', '4_features')
    BIO_DIR  = os.path.join(BASE, '0_data', '4_features')
    TARGETS_PATH = os.path.join(BASE, '0_data', 'game_targets.parquet')
    OUTPUT_DIR = os.path.join(BASE, '4_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATCAST_PATH = os.path.join(STAT_DIR, 'features_pitch15.parquet')
BIO_PATH      = os.path.join(BIO_DIR,  'features_bio_pitch15_full9.parquet')

# 실험 설정
KEYS = ['game_pk', 'pitcher', 'season']
TARGETS = ['y_whiff', 'y_xwoba', 'y_fip', 'y_era']
LOWER_BETTER = {'y_whiff': False, 'y_xwoba': True, 'y_fip': True, 'y_era': True}
TRAIN_SEASONS = [2021, 2022, 2023]
VAL_SEASON, TEST_SEASON = 2024, 2025
N_SEEDS = 30
CLIP_TARGETS = ['y_fip', 'y_era']   # 경기단위 극단값 → 1~99% 클리핑

for p in [STATCAST_PATH, BIO_PATH, TARGETS_PATH]:
    print(f'{"✅" if os.path.exists(p) else "❌"} {p}')

## 2. 데이터 병합 (정형 + 영상 + 4타겟)

정형 피처의 기존 y_whiff는 버리고 **game_targets의 4종으로 통일**.

In [ ]:
df_stat = pd.read_parquet(STATCAST_PATH)
df_bio  = pd.read_parquet(BIO_PATH)
tgt     = pd.read_parquet(TARGETS_PATH)

# 키 타입 통일
for d in (df_stat, df_bio, tgt):
    for k in KEYS:
        if k in d.columns:
            d[k] = d[k].astype('int64')

# 정형 feature 컬럼 (기존 타겟·메타 제거)
DROP = set(KEYS) | {'y_whiff', 'y_xwoba', 'y_fip', 'y_era'}
STAT_COLS = [c for c in df_stat.columns if c not in DROP]
BIO_COLS  = [c for c in df_bio.columns  if c not in (set(KEYS) | {'n_pitches_used'})]

# 병합: 정형 + 영상 (영상 매칭 경기만) + 타겟 4종
df = (df_stat[KEYS + STAT_COLS]
      .merge(df_bio[KEYS + BIO_COLS], on=KEYS, how='inner')
      .merge(tgt[KEYS + TARGETS],     on=KEYS, how='inner'))

# FIP/ERA 이상치 클리핑 (train 분포 기준으로 경계 산출 → 누수 방지)
tr_mask = df['season'].isin(TRAIN_SEASONS)
for c in CLIP_TARGETS:
    lo, hi = df.loc[tr_mask, c].quantile([0.01, 0.99])
    df[c] = df[c].clip(lo, hi)
    print(f'{c} 클리핑: [{lo:.2f}, {hi:.2f}]')

print(f'\n병합: {len(df):,}경기 | 정형 {len(STAT_COLS)} + 영상 {len(BIO_COLS)} feature')
print(f'시즌: {df["season"].value_counts().sort_index().to_dict()}')
ALL_COLS = STAT_COLS + BIO_COLS

## 3. 최적 파라미터

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    import subprocess; subprocess.run(['pip','install','-q','xgboost']); import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats
import json

# best_params 있으면 로드, 없으면 기본값
bp_path = os.path.join(OUTPUT_DIR, 'best_params.json')
if os.path.exists(bp_path):
    with open(bp_path) as f: BEST_PARAMS = json.load(f)
    BEST_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':0, 'early_stopping_rounds':50})
    print('best_params.json 로드')
else:
    BEST_PARAMS = {'n_estimators':300,'learning_rate':0.05,'max_depth':6,'subsample':0.8,
                   'colsample_bytree':0.8,'min_child_weight':1,'reg_alpha':0.0,'reg_lambda':1.0,
                   'random_state':42,'n_jobs':-1,'verbosity':0,'early_stopping_rounds':50}
    print('기본 파라미터 사용')
print(BEST_PARAMS)

## 4. 타겟별 정형 vs 융합 paired t-test

각 타겟마다: 결측 제거 → train/val/test 분할 → 30 seed로 정형·융합 학습 → val R² paired t-test.
정형/융합은 **동일 경기·동일 seed**로 학습해 공정 비교.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score

def run_target(y, n_seeds=N_SEEDS):
    d = df.dropna(subset=[y]).copy()                 # 타겟별 유효 경기
    tr = d[d['season'].isin(TRAIN_SEASONS)]
    vl = d[d['season'] == VAL_SEASON]
    te = d[d['season'] == TEST_SEASON]
    if min(len(tr), len(vl), len(te)) < 20:
        return None

    # baseline: train 평균으로 val 전부 예측했을 때 (멍청한 기준 모델)
    base_pred = tr[y].mean()
    base_rmse = mean_squared_error(vl[y], np.full(len(vl), base_pred)) ** 0.5
    base_mae  = mean_absolute_error(vl[y], np.full(len(vl), base_pred))

    # 분류 라벨(컨디션 좋음/나쁨): train 분포의 상/하위 33% 기준 (누수 방지: train 기준)
    q_lo, q_hi = tr[y].quantile([0.33, 0.67])
    def to_label(v):
        # whiff는 높을수록 좋음 / 나머지는 낮을수록 좋음
        good = (v >= q_hi) if not LOWER_BETTER[y] else (v <= q_lo)
        bad  = (v <= q_lo) if not LOWER_BETTER[y] else (v >= q_hi)
        return good.astype(int), bad.astype(int)  # 1=좋음, (중간은 평가서 제외)
    vl_good, vl_bad = to_label(vl[y])
    clf_mask = (vl_good == 1) | (vl_bad == 1)      # 중간 33% 제외하고 양극단만
    vl_bin = vl_good[clf_mask].values               # 1=좋음, 0=나쁨

    r2_stat, r2_fuse, rm_stat, rm_fuse, ma_stat, ma_fuse = [], [], [], [], [], []
    auc_stat, auc_fuse = [], []
    for seed in range(n_seeds):
        prm = {**BEST_PARAMS, 'random_state': seed}
        ms = xgb.XGBRegressor(**prm)
        ms.fit(tr[STAT_COLS], tr[y], eval_set=[(vl[STAT_COLS], vl[y])], verbose=False)
        ps = ms.predict(vl[STAT_COLS])
        mf = xgb.XGBRegressor(**prm)
        mf.fit(tr[ALL_COLS], tr[y], eval_set=[(vl[ALL_COLS], vl[y])], verbose=False)
        pf = mf.predict(vl[ALL_COLS])

        r2_stat.append(r2_score(vl[y], ps)); r2_fuse.append(r2_score(vl[y], pf))
        rm_stat.append(mean_squared_error(vl[y], ps)**0.5); rm_fuse.append(mean_squared_error(vl[y], pf)**0.5)
        ma_stat.append(mean_absolute_error(vl[y], ps));      ma_fuse.append(mean_absolute_error(vl[y], pf))

        # 분류 AUC: 회귀 예측값을 점수로 사용 (good=1 방향). whiff는 클수록 good → 그대로,
        # lower_better면 부호 반전해서 "클수록 good"으로 맞춤.
        sgn = -1.0 if LOWER_BETTER[y] else 1.0
        if vl_bin.sum() > 0 and vl_bin.sum() < len(vl_bin):
            auc_stat.append(roc_auc_score(vl_bin, sgn*ps[clf_mask.values]))
            auc_fuse.append(roc_auc_score(vl_bin, sgn*pf[clf_mask.values]))

    r2_stat, r2_fuse = np.array(r2_stat), np.array(r2_fuse)
    from scipy import stats as _st
    t, pv = _st.ttest_rel(r2_fuse, r2_stat)

    # test 성능(대표 seed=42, 정형)
    prm = {**BEST_PARAMS, 'random_state': 42}
    mt = xgb.XGBRegressor(**prm); mt.fit(tr[STAT_COLS], tr[y], eval_set=[(vl[STAT_COLS], vl[y])], verbose=False)
    pte = mt.predict(te[STAT_COLS])
    test_r2  = r2_score(te[y], pte)
    test_rmse= mean_squared_error(te[y], pte)**0.5

    mean = np.mean
    return {
        'target': y, 'n_games': len(d), 'n_tr': len(tr), 'n_vl': len(vl), 'n_te': len(te),
        'lower_better': LOWER_BETTER[y],
        # 회귀 (정형/융합)
        'stat_val_r2': r2_stat.mean(), 'fuse_val_r2': r2_fuse.mean(), 'diff_r2': r2_fuse.mean()-r2_stat.mean(),
        'stat_val_rmse': mean(rm_stat), 'fuse_val_rmse': mean(rm_fuse),
        'stat_val_mae': mean(ma_stat), 'fuse_val_mae': mean(ma_fuse),
        # baseline 대비 (평균찍기 RMSE/MAE) + skill score
        'base_rmse': base_rmse, 'base_mae': base_mae,
        'skill_rmse_stat': 1 - mean(rm_stat)/base_rmse,   # >0이면 평균보다 나음
        'skill_rmse_fuse': 1 - mean(rm_fuse)/base_rmse,
        # 분류 AUC (좋은날/나쁜날 구분력)
        'auc_stat': mean(auc_stat) if auc_stat else np.nan,
        'auc_fuse': mean(auc_fuse) if auc_fuse else np.nan,
        'n_clf': int(clf_mask.sum()),
        # test
        'test_r2_stat': test_r2, 'test_rmse_stat': test_rmse,
        't': t, 'p_raw': pv,
    }

results = [r for r in (run_target(y) for y in TARGETS) if r]
res = pd.DataFrame(results)

# 다중검정 보정 (Bonferroni)
m = len(res)
res['p_bonf'] = (res['p_raw'] * m).clip(upper=1.0)
res['sig_raw']  = res['p_raw']  < 0.05
res['sig_bonf'] = res['p_bonf'] < 0.05

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print('=== 회귀 지표 (정형 vs 융합) ===')
print(res[['target','n_vl','stat_val_r2','fuse_val_r2','diff_r2',
           'stat_val_rmse','stat_val_mae','base_rmse','skill_rmse_stat']].to_string(index=False))
print('\n=== 분류 AUC (좋은날/나쁜날 구분, 양극단 33%씩) ===')
print(res[['target','n_clf','auc_stat','auc_fuse']].to_string(index=False))
print('\n=== 영상 융합 검정 ===')
print(res[['target','diff_r2','p_raw','p_bonf','sig_bonf']].to_string(index=False))


## 5. 해석 + 저장

In [ ]:
print('=== 타겟별 종합 판정 ===')
for _, r in res.iterrows():
    # 예측가능성: R²>0.02 또는 skill>0 또는 AUC>0.55 중 하나라도
    predictable = (r['stat_val_r2'] > 0.02) or (r['skill_rmse_stat'] > 0) or (r['auc_stat'] > 0.55)
    base = '예측가능' if predictable else '예측난해'
    bio  = '영상 유의↑' if (r['sig_bonf'] and r['diff_r2'] > 0) else '영상 무의/음수'
    auc_s = f"{r['auc_stat']:.3f}" if pd.notna(r['auc_stat']) else 'NA'
    print(f"  {r['target']:8s} | R²={r['stat_val_r2']:+.4f} skill={r['skill_rmse_stat']:+.4f} "
          f"AUC={auc_s} ({base}) | 영상Δ={r['diff_r2']:+.4f} p_bonf={r['p_bonf']:.3f} → {bio}")

print('\n[지표 읽는 법]')
print(' - R²>0  : 평균예측보다 분산설명 나음 (음수면 평균만도 못함)')
print(' - skill>0: RMSE가 평균예측보다 작음 (개선율). R²와 같은 방향')
print(' - AUC>0.5: 좋은날/나쁜날 구분력 있음 (0.5=찍기, 0.6+=쓸만)')
print(' - 영상Δ>0 & sig_bonf: 영상이 통계적으로 도움 (지금까진 전부 음수/무의)')

# 주력 타겟: 예측가능 ∩ 도메인우선(whiff>xwoba>fip>era)
DOMAIN_RANK = {'y_whiff':0, 'y_xwoba':1, 'y_fip':2, 'y_era':3}
def is_pred(r): return (r['stat_val_r2']>0.02) or (r['skill_rmse_stat']>0) or (r['auc_stat']>0.55)
cand = res[res.apply(is_pred, axis=1)].copy()
if len(cand):
    cand['domain'] = cand['target'].map(DOMAIN_RANK)
    pick = cand.sort_values('domain').iloc[0]['target']
    print(f'\n추천 주력 타겟: {pick}')
else:
    print('\n⚠️ 예측가능 타겟 없음')

res.to_csv(os.path.join(OUTPUT_DIR, 'y_target_comparison.csv'), index=False)
print('저장:', os.path.join(OUTPUT_DIR, 'y_target_comparison.csv'))
res
